In [5]:
import os
import tarfile
import requests
import shutil
import pandas as pd

# Скачивание и разархивирование файлов
def download_and_extract(urls, extract_dir):
    os.makedirs(extract_dir, exist_ok=True)

    for url in urls:
        filename = os.path.join(extract_dir, url.split("/")[-1])

        # Скачивание файла
        print(f"Downloading {url}...")
        response = requests.get(url, stream=True)
        with open(filename, 'wb') as file:
            shutil.copyfileobj(response.raw, file)
        print(f"Downloaded {filename}")

        # Разархивирование файла
        if filename.endswith("tar.bz2"):
            print(f"Extracting {filename}...")
            with tarfile.open(filename, "r:bz2") as tar:
                tar.extractall(path=extract_dir)
            print(f"Extracted {filename}")

# Чтение писем из директории и создание DataFrame
def emails_to_dataframe(directory, label):
    emails = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            filepath = os.path.join(root, file)
            with open(filepath, 'r', errors='ignore') as email_file:
                emails.append({"text": email_file.read(), "label": label})

    return pd.DataFrame(emails)

# Пути для скачивания данных
DOWNLOAD_HAM = [
    "https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2",
    "https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham_2.tar.bz2",
    "https://spamassassin.apache.org/old/publiccorpus/20030228_hard_ham.tar.bz2"
]

DOWNLOAD_SPAM = [
    "https://spamassassin.apache.org/old/publiccorpus/20050311_spam_2.tar.bz2",
    "https://spamassassin.apache.org/old/publiccorpus/20030228_spam.tar.bz2"
]

# Директории для извлечения
HAM_DIR = "ham_emails"
SPAM_DIR = "spam_emails"

# Скачивание и разархивирование данных
download_and_extract(DOWNLOAD_HAM, HAM_DIR)
download_and_extract(DOWNLOAD_SPAM, SPAM_DIR)

# Преобразование писем в DataFrame
df_ham = emails_to_dataframe(HAM_DIR, "ham")
df_spam = emails_to_dataframe(SPAM_DIR, "spam")

# Объединение данных в один DataFrame
df_all = pd.concat([df_ham, df_spam], ignore_index=True)

# Выводим первые несколько строк DataFrame


Downloaded ham_emails/20030228_easy_ham.tar.bz2
Extracting ham_emails/20030228_easy_ham.tar.bz2...
Extracted ham_emails/20030228_easy_ham.tar.bz2
Downloaded ham_emails/20030228_easy_ham_2.tar.bz2
Extracting ham_emails/20030228_easy_ham_2.tar.bz2...
Extracted ham_emails/20030228_easy_ham_2.tar.bz2
Downloaded ham_emails/20030228_hard_ham.tar.bz2
Extracting ham_emails/20030228_hard_ham.tar.bz2...
Extracted ham_emails/20030228_hard_ham.tar.bz2
Downloaded spam_emails/20050311_spam_2.tar.bz2
Extracting spam_emails/20050311_spam_2.tar.bz2...
Extracted spam_emails/20050311_spam_2.tar.bz2
Downloaded spam_emails/20030228_spam.tar.bz2
Extracting spam_emails/20030228_spam.tar.bz2...
Extracted spam_emails/20030228_spam.tar.bz2
                                                text label
0  BZh91AY&SYoMa 0$T   Hc$Mv:k@` (_fn7Yݎ{n...   ham
1  BZh91AY&SY%)_\n۾}&<*c~=dSv˛v]vv-q TT\{{1...   ham
2  BZh91AY&SY"M|G|c3~ܢ\tdǃ}6@) }#d@]ݴvzPgu...   ham
3  From exmh-workers-admin@redhat.com  W

In [6]:
df_all

,text,label
0,BZh91AY&SYoMa 0$T   Hc$Mv:k@` (_fn7Yݎ{n...,ham
1,BZh91AY&SY %)_\n۾}&<*c~=dSv˛v]vv- q TT\{{1...,ham
2,"BZh91AY&SY""M|G|c3~ܢ\t dǃ} 6@) }#d@]ݴ vzPgu...",ham
3,From exmh-workers-admin@redhat.com Wed Aug 28...,ham
4,From pudge@perl.org Mon Dec 2 11:10:55 2002\...,ham
...,...,...
6051,From IKE_EJOH@YAHOO.COM Thu Aug 29 15:40:40 2...,spam
6052,From calebn@msn.com Thu Sep 19 13:26:56 2002\...,spam
6053,From ferdinand@caramail.com Tue Sep 3 18:16:...,spam
6054,From nlv@insiq.us Thu Sep 5 11:46:24 2002\nR...,spam


In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [15]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def process_text(text):
  tokens = word_tokenize(text)
  lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
  filtered_tokens = [token for token in lemmatized_tokens if token.lower() not in stop_words]
  return ' '.join(lemmatized_tokens)

df_all['text'] = df_all['text'].apply(process_text)
df_all

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,text,label
0,BZh91AY & SYoMa 0 $ T   Hc $ Mv : k @ ` ...,ham
1,BZh91AY & SY %  ) _ ۾ } & < * c~=dSv˛v ] vv...,ham
2,BZh91AY & SY '' M|G|c3~ܢ dǃ } 6 @ ) } # d @ ]...,ham
3,From exmh-workers-admin @ redhat.com Wed Aug 2...,ham
4,From pudge @ perl.org Mon Dec 2 11:10:55 2002 ...,ham
...,...,...
6051,From IKE_EJOH @ YAHOO.COM Thu Aug 29 15:40:40 ...,spam
6052,From calebn @ msn.com Thu Sep 19 13:26:56 2002...,spam
6053,From ferdinand @ caramail.com Tue Sep 3 18:16:...,spam
6054,From nlv @ insiq.us Thu Sep 5 11:46:24 2002 Re...,spam


In [17]:
# Разделение данных на обучающий и тестовый наборы
X_train, X_test, y_train, y_test = train_test_split(df_all['text'], df_all['label'], test_size=0.2, random_state=42)

# Преобразование текстовых данных в векторы TF-IDF
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)

tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)

# Обучение модели логистической регрессии
clf = LogisticRegression(max_iter=100)
clf.fit(X_train_tfidf, y_train)

# Преобразование тестовых данных в векторы TF-IDF
X_test_counts = vectorizer.transform(X_test)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

# Предсказание меток для тестовых данных
y_pred = clf.predict(X_test_tfidf)

# Оценка точности модели
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Вывод отчета о классификации и матрицы ошибок
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.97
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       846
        spam       0.97      0.92      0.94       366

    accuracy                           0.97      1212
   macro avg       0.97      0.95      0.96      1212
weighted avg       0.97      0.97      0.97      1212

[[834  12]
 [ 29 337]]
